# Lab 2 — Predicción de Resistencia a la Compresión — Vía IA-asistida

**Sesión:** 1 o 3 · Python, EDA y aprendizaje supervisado básico

**Público:** ingenieros civiles **sin experiencia en programación** · **Entorno:** GitHub Codespaces o `labs/.venv`

## Cómo trabajar (Copilot, Gemini, Cursor, etc.)

1. Abre este repo en **Codespaces** o activa `source labs/.venv/bin/activate`.
2. En cada sección: lee el **objetivo**, copia el **prompt sugerido** a tu asistente de IA.
3. Pega **solo** el código generado en la celda `### PEGA AQUÍ EL CÓDIGO DE LA IA ###`.
4. Ejecuta la celda y la **Autoevaluación**; busca ✅ antes de avanzar.
5. Registra tus prompts en [`prompts_entregados.md`](prompts_entregados.md) (entrega obligatoria en esta vía).
6. Referencia docente: `resistencia_compresion_solucion_ia.ipynb` (no distribuir al inicio).

> **Vía manual (hiperparámetros):** usa `{nombre_base}.ipynb` si prefieres editar variables sin IA.

**La IA propone; tú validas** con autoevaluación, gráficos y sentido físico/estructural.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_tipo_problema, verificar_carga, verificar_columna,
    verificar_resumen, verificar_umbral_resistencia, verificar_correlaciones,
    verificar_scatter_filtro, verificar_split, verificar_comparacion_regresion,
    verificar_clasificacion, verificar_importancias, resumen_seccion,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, accuracy_score, classification_report, confusion_matrix
from xgboost import XGBRegressor, XGBClassifier

%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Entorno listo (Codespaces / labs/.venv).')

## Contexto del dataset (UCI)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Cemento, Escoria, CenizaVolante | kg/m³ | Dosificación de ligantes y adiciones |
| Agua | kg/m³ | Relación agua/cemento (W/C) |
| Superplastificante | kg/m³ | Reducir agua sin perder trabajabilidad |
| AgregadoGrueso / Fino | kg/m³ | Grava y arena |
| Edad | días | Curado antes del ensayo |
| **Resistencia** | **MPa** | **Variable a predecir (y)** |

Fuente: Prof. I-Cheng Yeh · 1 030 mezclas · **regresión** (MPa) y **clasificación** (fuerte/débil).

Objetivo del lab: explorar el dataset al máximo — EDA, comparar modelos de regresión (sklearn + XGBoost) y un modelo de clasificación con el umbral de la sección 5.

Detalle ampliado: [`data/DATOS.md`](data/DATOS.md).

## Pregunta 1 — Contexto del hormigón y Machine Learning

### 📘 Subpreguntas
- **1.a** ¿Predecir resistencia en MPa es **regresión** o **clasificación**?
- **1.b** ¿Por qué usamos MPa y no una etiqueta «bueno/malo»?

En planta, un modelo puede **explorar dosificaciones** antes de ensayos destructivos costosos.

#### ✍️ Tu respuesta (opcional, 2–3 líneas)



### 🤖 Sección 1 — Contexto ML

**Objetivo:** Identificar que predecir MPa es un problema de regresión.

**Lista de tareas**
1. Definir la variable TIPO_PROBLEMA con el valor correcto.
2. Imprimir el tipo elegido.
3. Ejecutar autoevaluación hasta ver ✅.

**Prompt sugerido** (copiar al asistente):

```text
Estoy en un Jupyter Lab de ingeniería civil (predicción de resistencia del hormigón).
En la celda marcada ### PEGA AQUÍ EL CÓDIGO DE LA IA ### solo necesito:
- Asignar TIPO_PROBLEMA = "regresion" o "clasificacion" según si el target es MPa continuo.
- Un print del valor.
No uses otros imports. El target es Resistencia en MPa (1030 mezclas UCI).
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._

In [ ]:
### PEGA AQUÍ EL CÓDIGO DE LA IA ###
# Indica el tipo de problema de ML para este lab.
TIPO_PROBLEMA = "regresion"  # Prueba cambiar a "clasificacion" y revisa la autoevaluación

print(f"Tipo de problema elegido: {TIPO_PROBLEMA}")

In [ ]:
# --- Autoevaluación 1 ---
r = verificar_tipo_problema(TIPO_PROBLEMA)
resumen_seccion('1 — Contexto ML', r)

## Pregunta 2 — Carga del dataset

### 📘 Subpreguntas
- **2.a** ¿Cuántas **features** hay frente a 1 **target**?
- **2.b** ¿Por qué convertimos el XLS UCI a CSV en español?

In [ ]:
# --- PRE-ESCRITO: carga desde data/concrete.csv ---
RUTA_DATOS = Path("data/concrete.csv")
df = pd.read_csv(RUTA_DATOS)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} filas × {df.shape[1]} columnas")

### 🤖 Sección 2 — Carga del dataset

**Objetivo:** Mostrar las primeras filas del CSV ya cargado en df.

**Lista de tareas**
1. Definir N_FILAS_HEAD entre 1 y 20.
2. Mostrar df.head(N_FILAS_HEAD) con display.
3. Verificar ✅ en autoevaluación 2.

**Prompt sugerido** (copiar al asistente):

```text
En Jupyter ya existe df = pd.read_csv("data/concrete.csv") con 1030 filas y 9 columnas en español.
Solo en la celda ### PEGA AQUÍ EL CÓDIGO DE LA IA ###:
- N_FILAS_HEAD = 5 (o entre 1 y 20)
- print y display(df.head(N_FILAS_HEAD))
No vuelvas a leer el CSV.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._

In [ ]:
### PEGA AQUÍ EL CÓDIGO DE LA IA ###
N_FILAS_HEAD = 5  # Prueba 3, 8 o 10

print(f"Primeras {N_FILAS_HEAD} mezclas:")
display(df.head(N_FILAS_HEAD))

In [ ]:
# --- Autoevaluación 2 ---
r = verificar_carga(df, N_FILAS_HEAD)
resumen_seccion('2 — Carga', r)
print("ℹ️ 8 features + 1 target (Resistencia).")

## Pregunta 3 — Calidad de datos

### 📘 Subpreguntas
- **3.a** ¿Hay valores nulos en el dataset?
- **3.b** Si fueras jefe de planta, ¿qué columna revisarías primero y por qué?

### 🤖 Sección 3 — Calidad de datos

**Objetivo:** Revisar estadísticas de una columna de dosificación.

**Lista de tareas**
1. Elegir COLUMNA_REVISAR entre columnas del df (ej. Agua, Edad, Cemento).
2. Calcular y mostrar describe() de esa columna.
3. Lograr ✅ en autoevaluación 3.

**Prompt sugerido** (copiar al asistente):

```text
df ya está cargado (concrete.csv, columnas: Cemento, Escoria, CenizaVolante, Agua, Superplastificante, AgregadoGrueso, AgregadoFino, Edad, Resistencia).
Celda ### PEGA AQUÍ EL CÓDIGO DE LA IA ###:
- COLUMNA_REVISAR = "Agua" (u otra columna válida)
- stats_col = df[COLUMNA_REVISAR].describe()
- display(stats_col)
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._

In [ ]:
### PEGA AQUÍ EL CÓDIGO DE LA IA ###
COLUMNA_REVISAR = "Agua"  # Prueba "Edad" o "Cemento"

stats_col = df[COLUMNA_REVISAR].describe()
print(f"Estadísticas de «{COLUMNA_REVISAR}»:")
display(stats_col)

In [ ]:
# --- Autoevaluación 3 ---
r = verificar_columna(df, COLUMNA_REVISAR)
resumen_seccion('3 — Calidad', r)

## Pregunta 4 — Estadísticas descriptivas

### 📘 Subpreguntas
- **4.a** Mirando `describe()`, ¿qué variable tiene mayor dispersión relativa (std vs media)?
- **4.b** ¿Qué implicación tiene para el modelado?

### 🤖 Sección 4 — Describe

**Objetivo:** Resumen estadístico de columnas elegidas.

**Lista de tareas**
1. Definir COLUMNAS_RESUMEN con al menos Agua, Edad y Resistencia.
2. Mostrar df[COLUMNAS_RESUMEN].describe().
3. ✅ en autoevaluación 4.

**Prompt sugerido** (copiar al asistente):

```text
Usar df existente. En ### PEGA AQUÍ EL CÓDIGO DE LA IA ###:
COLUMNAS_RESUMEN = ["Agua", "Edad", "Resistencia"]
resumen = df[COLUMNAS_RESUMEN].describe()
display(resumen)
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._

In [ ]:
### PEGA AQUÍ EL CÓDIGO DE LA IA ###
COLUMNAS_RESUMEN = ["Agua", "Edad", "Resistencia"]  # Quita o agrega columnas

resumen = df[COLUMNAS_RESUMEN].describe()
display(resumen)

In [ ]:
# --- Autoevaluación 4 ---
r = verificar_resumen(resumen, COLUMNAS_RESUMEN)
resumen_seccion('4 — Describe', r)

## Pregunta 5 — Distribución del target (Resistencia)

### 📘 Subpreguntas
- **5.a** ¿Hay más mezclas por debajo o por encima de 40 MPa?
- **5.b** ¿Qué sesgo visual ves en el histograma?

In [ ]:
# --- PRE-ESCRITO: histograma base ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['Resistencia'], bins=30, color='#3498db', edgecolor='white')
ax.set_xlabel('Resistencia (MPa)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de resistencia a compresión')
plt.tight_layout()
plt.show()

### 🤖 Sección 5 — Umbral de resistencia

**Objetivo:** Contar mezclas fuertes/débiles según umbral en MPa.

**Lista de tareas**
1. Definir UMBRAL_RESISTENCIA (ej. 40 MPa).
2. Calcular n_fuertes y n_debiles.
3. Imprimir conteos y pasar autoevaluación 5.

**Prompt sugerido** (copiar al asistente):

```text
df tiene columna Resistencia (MPa). Celda ### PEGA AQUÍ ###:
UMBRAL_RESISTENCIA = 40
n_fuertes = int((df['Resistencia'] >= UMBRAL_RESISTENCIA).sum())
n_debiles = int((df['Resistencia'] < UMBRAL_RESISTENCIA).sum())
print conteos con f-string
```

**Prompt alternativo válido** (misma sección, otros parámetros permitidos):

```text
Usa UMBRAL_RESISTENCIA = 35 o 45; los conteos deben ser coherentes con el histograma.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._

In [ ]:
### PEGA AQUÍ EL CÓDIGO DE LA IA ###
UMBRAL_RESISTENCIA = 40  # MPa — experimenta: 30, 35, 45

n_fuertes = int((df['Resistencia'] >= UMBRAL_RESISTENCIA).sum())
n_debiles = int((df['Resistencia'] < UMBRAL_RESISTENCIA).sum())
print(f"≥ {UMBRAL_RESISTENCIA} MPa: {n_fuertes} | < {UMBRAL_RESISTENCIA} MPa: {n_debiles}")

In [ ]:
# --- Autoevaluación 5 ---
r = verificar_umbral_resistencia(df, UMBRAL_RESISTENCIA, n_fuertes, n_debiles)
resumen_seccion('5 — Target', r)

## Pregunta 6 — Correlación entre variables

### 📘 Subpreguntas
- **6.a** ¿Qué ingrediente correlaciona **más** con Resistencia (valor absoluto)?
- **6.b** ¿El signo de Agua coincide con la física del hormigón?

In [ ]:
# --- PRE-ESCRITO: matriz de correlación ---
corr = df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlación — dosificación vs resistencia')
plt.tight_layout()
plt.show()

### 🤖 Sección 6 — Correlación

**Objetivo:** Listar top correlaciones con Resistencia y validar signo de Agua.

**Lista de tareas**
1. TOP_N_CORR = 3 (o 4-5).
2. Ordenar |r| respecto a Resistencia e imprimir.
3. Agua debe correlacionar negativamente.

**Prompt sugerido** (copiar al asistente):

```text
Ya existe corr = df.corr(numeric_only=True). En ### PEGA AQUÍ ###:
TOP_N_CORR = 3
corr_target = corr['Resistencia'].drop('Resistencia')
top_corr = corr_target.abs().sort_values(ascending=False).head(TOP_N_CORR).index.tolist()
imprimir r de cada nombre en top_corr
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._

In [ ]:
### PEGA AQUÍ EL CÓDIGO DE LA IA ###
TOP_N_CORR = 3  # Prueba 4 o 5

corr_target = corr['Resistencia'].drop('Resistencia')
top_corr = corr_target.abs().sort_values(ascending=False).head(TOP_N_CORR).index.tolist()
print("Top correlaciones con Resistencia (|r|):")
for nombre in top_corr:
    print(f"  {nombre}: r = {corr_target[nombre]:.3f}")

In [ ]:
# --- Autoevaluación 6 ---
from _verificar import verificar
r = verificar_correlaciones(top_corr, TOP_N_CORR)
r.append(
    verificar(
        corr_target['Agua'] < 0,
        "✅ Agua correlaciona negativamente con resistencia (física coherente).",
        "❌ Agua debería tener correlación negativa con resistencia.",
    )
)
resumen_seccion('6 — Correlación', r)

## Pregunta 7 — Relación física Agua vs Resistencia

### 📘 Subpreguntas
- **7.a** ¿Qué patrón ves en el scatter?
- **7.b** ¿Por qué más agua suele **bajar** la resistencia?

In [ ]:
# --- PRE-ESCRITO: scatter con filtro parametrizable ---
def graficar_agua_resistencia(datos, titulo_extra=''):
    fig, ax = plt.subplots(figsize=(8, 5))
    sc = ax.scatter(
        datos['Agua'], datos['Resistencia'],
        c=datos['Edad'], cmap='viridis', alpha=0.7, edgecolors='white', linewidth=0.3,
    )
    ax.set_xlabel('Agua (kg/m³)')
    ax.set_ylabel('Resistencia (MPa)')
    ax.set_title(f'Agua vs Resistencia {titulo_extra}')
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Edad (días)')
    plt.tight_layout()
    plt.show()

### 🤖 Sección 7 — Agua vs Resistencia

**Objetivo:** Filtrar por edad mínima de curado y graficar scatter.

**Lista de tareas**
1. EDAD_MIN_DIAS (7, 14, 28 o 56).
2. Filtrar df y llamar graficar_agua_resistencia(df_filtrado, título).
3. n_filtradas = len(df_filtrado).

**Prompt sugerido** (copiar al asistente):

```text
Función graficar_agua_resistencia ya está definida arriba. En ### PEGA AQUÍ ###:
EDAD_MIN_DIAS = 28
df_filtrado = df[df['Edad'] >= EDAD_MIN_DIAS]
n_filtradas = len(df_filtrado)
graficar_agua_resistencia(df_filtrado, f"(Edad ≥ {EDAD_MIN_DIAS} días, n={n_filtradas})")
```

**Prompt alternativo válido** (misma sección, otros parámetros permitidos):

```text
EDAD_MIN_DIAS = 14 si quieres más puntos; debe quedar n_filtradas > 0.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._

In [ ]:
### PEGA AQUÍ EL CÓDIGO DE LA IA ###
EDAD_MIN_DIAS = 28  # Prueba 7, 14 o 56 (días de curado mínimo)

df_filtrado = df[df['Edad'] >= EDAD_MIN_DIAS]
n_filtradas = len(df_filtrado)
graficar_agua_resistencia(df_filtrado, f"(Edad ≥ {EDAD_MIN_DIAS} días, n={n_filtradas})")

In [ ]:
# --- Autoevaluación 7 ---
r = verificar_scatter_filtro(n_filtradas, EDAD_MIN_DIAS)
resumen_seccion('7 — Agua vs Resistencia', r)

## Pregunta 8 — Partición train / test

Puente desde Lab 0: `X` = features, `y` = Resistencia, luego `model.fit(X_train, y_train)`.

### 📘 Subpreguntas
- **8.a** ¿Para qué sirve `random_state=42`?
- **8.b** ¿Por qué no evaluamos el modelo con los mismos datos de entrenamiento?

In [ ]:
# --- PRE-ESCRITO: definir X e y ---
TARGET = "Resistencia"
FEATURES_TODAS = [c for c in df.columns if c != TARGET]
X = df[FEATURES_TODAS]
y = df[TARGET]
print(f"X: {X.shape} | y: {y.shape}")

### 🤖 Sección 8 — Train/test

**Objetivo:** Particionar X,y con train_test_split.

**Lista de tareas**
1. TEST_SIZE = 0.2 y RANDOM_STATE = 42 para pasar verificador.
2. Imprimir tamaños train y test.

**Prompt sugerido** (copiar al asistente):

```text
X, y ya definidos (target Resistencia). ### PEGA AQUÍ ###:
TEST_SIZE = 0.2
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print len train y test
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._

In [ ]:
### PEGA AQUÍ EL CÓDIGO DE LA IA ###
TEST_SIZE = 0.2       # Prueba 0.15 o 0.25
RANDOM_STATE = 42     # Cambia y observa que varían las métricas

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# --- Autoevaluación 8 ---
r = verificar_split(len(X_train), len(X_test), TEST_SIZE, RANDOM_STATE)
resumen_seccion('8 — Train/Test', r)

## Pregunta 9 — Comparar modelos de regresión

Compara **tres** enfoques para predecir MPa: un modelo lineal (baseline), Random Forest y XGBoost.

### 📘 Subpreguntas
- **9.a** ¿Qué modelo obtiene mayor R² en test y por qué los árboles suelen ganar aquí?
- **9.b** ¿Qué pasa con R² si quitas `Edad` de `COLUMNAS_X`?


### 🤖 Sección 9 — Comparar regresión

**Objetivo:** Entrenar LinearRegression, RandomForestRegressor y XGBRegressor con las mismas features; comparar R² en test.

**Lista de tareas**
1. COLUMNAS_X = FEATURES_TODAS (incluye Edad).
2. Guardar `resultados_r2` como dict nombre → R².
3. Elegir el mejor modelo y guardar `r2_test`, `y_pred`.

**Prompt sugerido** (copiar al asistente):

```text
En Jupyter tengo X_train, X_test, y_train, y_test, FEATURES_TODAS, RANDOM_STATE.
En ### PEGA AQUÍ ### compara 3 regresores:
- LinearRegression()
- RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
- XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=RANDOM_STATE)
COLUMNAS_X = FEATURES_TODAS
resultados_r2 = dict con R² de cada uno en test
Muestra tabla ordenada y barplot horizontal de R²
mejor_modelo_nombre, r2_test, y_pred del ganador
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### PEGA AQUÍ EL CÓDIGO DE LA IA ###
COLUMNAS_X = FEATURES_TODAS  # Quita 'Edad' y vuelve a ejecutar

X_train_sel = X_train[COLUMNAS_X]
X_test_sel = X_test[COLUMNAS_X]

MODELOS_REGRESION = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators=100, max_depth=10, random_state=RANDOM_STATE
    ),
    "XGBoost": XGBRegressor(
        n_estimators=100, max_depth=6, learning_rate=0.1, random_state=RANDOM_STATE
    ),
}

resultados_r2 = {}
predicciones_reg = {}
for nombre, modelo in MODELOS_REGRESION.items():
    modelo.fit(X_train_sel, y_train)
    y_hat = modelo.predict(X_test_sel)
    resultados_r2[nombre] = r2_score(y_test, y_hat)
    predicciones_reg[nombre] = y_hat

tabla_r2 = pd.Series(resultados_r2).sort_values(ascending=False)
mejor_modelo_nombre = tabla_r2.index[0]
modelo_reg = MODELOS_REGRESION[mejor_modelo_nombre]
y_pred = predicciones_reg[mejor_modelo_nombre]
r2_test = resultados_r2[mejor_modelo_nombre]

print("Comparación R² (test):")
display(tabla_r2.to_frame("R²"))

fig, ax = plt.subplots(figsize=(7, 4))
tabla_r2.sort_values().plot(kind="barh", ax=ax, color="#3498db")
ax.set_xlabel("R² en test")
ax.set_title("Regresión — comparación de modelos")
plt.tight_layout()
plt.show()

print(f"Mejor modelo: {mejor_modelo_nombre} (R² = {r2_test:.3f})")

In [ ]:
# --- Autoevaluación 9 ---
r = verificar_comparacion_regresion(resultados_r2, COLUMNAS_X)
resumen_seccion('9 — Comparar regresión', r)
print("ℹ️ R² sin Edad cae fuerte (~0.39): el curado explica mucha variabilidad.")

## Pregunta 10 — Clasificación: mezcla fuerte o débil

Usa el umbral de la sección 5 para predecir si la mezcla supera **40 MPa** (clasificación binaria).

### 📘 Subpreguntas
- **10.a** ¿Por qué a veces conviene clasificar aunque tengamos MPa continuos?
- **10.b** ¿Qué variable importa más para separar fuerte/débil?


### 🤖 Sección 10 — Clasificación binaria

**Objetivo:** Crear etiqueta fuerte/débil con UMBRAL_RESISTENCIA, entrenar RandomForestClassifier y evaluar.

**Lista de tareas**
1. `y_clf = (df['Resistencia'] >= UMBRAL_RESISTENCIA).astype(int)`
2. `train_test_split` con `stratify=y_clf`, mismo TEST_SIZE y RANDOM_STATE.
3. Accuracy, matriz de confusión e importancias.

**Prompt sugerido** (copiar al asistente):

```text
En Jupyter: df, UMBRAL_RESISTENCIA, COLUMNAS_X, TEST_SIZE, RANDOM_STATE.
Clasificación binaria fuerte/débil:
- y_clf desde Resistencia >= umbral
- RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE)
- acc_test, confusion_matrix heatmap, top 5 importancias en barh
- importancias_ordenadas = lista de nombres top
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
### PEGA AQUÍ EL CÓDIGO DE LA IA ###
y_clf = (df[TARGET] >= UMBRAL_RESISTENCIA).astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X[COLUMNAS_X], y_clf, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_clf
)

modelo_clf = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=RANDOM_STATE
)
modelo_clf.fit(X_train_c, y_train_c)
y_pred_clf = modelo_clf.predict(X_test_c)
acc_test = accuracy_score(y_test_c, y_pred_clf)

importancias_clf = dict(zip(COLUMNAS_X, modelo_clf.feature_importances_))
N_TOP_IMPORTANCIAS = 5
importancias_ordenadas = (
    pd.Series(importancias_clf).sort_values(ascending=False).head(N_TOP_IMPORTANCIAS).index.tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test_c, y_pred_clf)
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
    xticklabels=["debil", "fuerte"], yticklabels=["debil", "fuerte"],
)
axes[0].set_xlabel("Predicho")
axes[0].set_ylabel("Real")
axes[0].set_title(f"Clasificación (acc = {acc_test:.3f})")

pd.Series(importancias_clf).sort_values().tail(N_TOP_IMPORTANCIAS).plot(
    kind="barh", ax=axes[1], color="#e74c3c"
)
axes[1].set_title(f"Top {N_TOP_IMPORTANCIAS} — importancia (clasificación)")
axes[1].set_xlabel("Importancia relativa")

plt.tight_layout()
plt.show()

print(classification_report(y_test_c, y_pred_clf, target_names=["debil", "fuerte"], digits=3))
print("Top importancias (clasificación):", importancias_ordenadas)

In [ ]:
# --- Autoevaluación 10 ---
r = verificar_clasificacion(acc_test, UMBRAL_RESISTENCIA, importancias_clf)
r.extend(verificar_importancias(importancias_ordenadas, N_TOP_IMPORTANCIAS))
resumen_seccion('10 — Clasificación', r)

## Cierre profesional

### ✍️ Reflexión final (3 frases)
- El mejor modelo de regresión fue ___ porque ___.
- Para clasificar fuerte/débil usaría el umbral ___ MPa porque ___.
- Antes de obra real validaría con ___.

---
**Checklist:** 10 autoevaluaciones en ✅ · gráficos revisados · [`data/DATOS.md`](data/DATOS.md) leído.
